In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("Econ126_Homework_06.ipynb")

In [ ]:
import numpy as np
import pandas as pd
import linearsolve as ls
import matplotlib.pyplot as plt
plt.style.use('classic')
plt.rcParams['figure.facecolor'] = 'white'

# Homework 6

**Instructions:** Complete the notebook below. Download the completed notebook in HTML format. Upload assignment using Canvas.

## Exercise: Four-Period Cake-Eating Problem

Now consider a the four-period cake-eating problem. A person has an initial quantity of cake $K_0>0$ and receives utility from consuming cake in period 0, period 1, period 2, and period 3:

\begin{align}
U(C_0,C_1,C_2,C_3) & = \log C_0 + \beta \log C_1+ \beta^2 \log C_2 + \beta^3 \log C_3.
\end{align}

The person chooses how much to consume in each period, $C_0$, $C_1$, $C_2$, $C_3$, and quantities of cake $K_1$, $K_2$, and $K_3$ to carry foward from period 0, period 1, and period 2 to maximize her utility subject to the following two budget constraints:

\begin{align}
C_0 + K_1 & = K_0\\
C_1 + K_2 & = K_1\\
C_2 + K_3 & = K_2\\
C_3 & = K_3,
\end{align}

where the boundary condition $K_4 = 0$ has been imposed.

### Part (a)

Find the first-order conditions with respect to $K_1$, $K_2$, and $K_3$ for the four-period cake-eating problem. Then solve by hand for:

1. $K_3$ as a function of $K_2$
1. $K_2$ as a function of $K_1$
2. $K_1$ as a function of $K_0$

*There is nothing to submit for this question* Use what you derive in this part to do **Part (b)**, **Part (c)**, and **Part (d)**.

### Part (b)

Suppose that the initial quantity of cake is $K_0 = 100$. For $\beta = 0, 0.25, 0.5, 0.75, 0.99, 1, 2$, plot the optimal path for the quantity of cake for periods 0 through 4. Add labels to the plot command so you know which line corresponds to which value of $\beta$.

<!-- BEGIN QUESTION -->



In [ ]:
...

<!-- END QUESTION -->

### Part (c)

Suppose that the initial quantity of cake is $K_0 = 100$. For $\beta = 0, 0.25, 0.5, 0.75, 0.99, 1, 2$, plot the optimal path for cake consumption for periods 0 through 3. Add labels to the plot command so you know which line corresponds to which value of $\beta$.

<!-- BEGIN QUESTION -->



In [ ]:
...

<!-- END QUESTION -->

### Part (d)

Suppose that $\beta=0.95$. For $K_0 = 1, 10, 20, 50$, plot the optimal path for the quantity of cake for periods 0 through 4. Add labels to the plot command so you know which line corresponds to which value of $K_0$.

<!-- BEGIN QUESTION -->



In [ ]:
...

<!-- END QUESTION -->

## Exercise: The Stochastic Solow Growth Model

In class we simulated the stochastic Solow growth model with three endogenous variables: $K_t$, $Y_t$, and $A_t$. Now, reconsider the model with two additional variables, consumption $C_t$ and investment $I_t$, and two additional equations:

\begin{align}
Y_t & = A_t K_t^{\alpha}\\
C_{t} & = (1-s)Y_t\\
Y_{t} & = C_t + I_t\\
K_{t+1} & = sY_t + (1-\delta) K_t\\
\log A_{t+1} & = \rho \log A_t + \epsilon_{t+1} 
\end{align}

where $\epsilon_{t+1} \sim \mathcal{N}(0,\sigma^2)$. In this exercise, you will construct a 401 period (10-year) simulation of the model.

For the other parameter, use the following values for the simulation:

| $$\rho$$ | $$\sigma$$ | $$s$$  | $$\alpha$$ | $$\delta $$ | $$T$$  |
|----------|------------|--------|------------|-------------|--------|
| 0.8      | 0.005      | 0.12   | 0.33       |  0.025      | 401    |

In [ ]:
# Create a variable called 'parameters' that is a Pandas Series containing the parameter values for the simulation
parameters = pd.Series(dtype=float)
parameters['rhoa'] = .8
parameters['sigma_squared'] = 0.005**2
parameters['alpha'] = 0.33
parameters['delta'] = 0.025
parameters['s'] = 0.12

# Display the model's parameters
display(parameters)

In [ ]:
# Define a function that evaluates the equilibrium conditions.
def equilibrium_equations(variables_forward,variables_current,parameters):
    
    # Parameters.
    p = parameters
    
    # Current variables.
    cur = variables_current
    
    # Forward variables.
    fwd = variables_forward
    
    # Production function
    prod_fn = cur.a*cur.k**p.alpha - cur.y
    
    # Capital evolution
    capital_evolution = p.s*cur.a*cur.k**p.alpha + (1 - p.delta)*cur.k - fwd.k
    
    # Consumption function
    consumption_function = (1-p.s)*cur.y - cur.c
    
    # Market clearing
    market_clearing = cur.c + cur.i - cur.y
    
    # Exogenous tfp
    tfp_proc = p.rhoa*np.log(cur.a) - np.log(fwd.a)
    
    # Stack equilibrium conditions into a numpy array
    return np.array([
        prod_fn,
        capital_evolution,
        consumption_function,
        market_clearing,
        tfp_proc
        ])

In [ ]:
# Initialize the model.
model = ls.model(equations = equilibrium_equations,
                 exo_states=['a'],
                 endo_states=['k'],
                 costates=['y','c','i'],
                 parameters = parameters)

In [ ]:
# Compute the steady state numerically using the .compute_ss() method.
guess = pd.Series({
    'a':1,
    'k':4,
    'y':1,
    'c':1,
    'i':1
    })
model.compute_ss(guess)

In [ ]:
# Use the .approximate_and_solve() method to compute the log-linear approximation of the model around the 
# non-stochastic steady state and solve endogenous variables in terms of state variables.
model.approximate_and_solve()

<!-- BEGIN QUESTION -->



In [ ]:
# Use .stoch_sim() method on the variable `model` to compute stochastic simulation with the following arguments:

#    1. seed = 126
#    2. T = 401
#    3. variances= [parameters['sigma_squared'] or variances = [0.005**2]
model.stoch_sim(seed=126,T=401,variances= parameters['sigma_squared'])

# Display first 10 rows of model.simulated
...

In [ ]:
# Plot the computed stochastic simulation results. Creae two axes side-by-side.
#     1. Left axes: plot: a, k, i, y, c
#     1. Right axes: plot: shock to TFP
...

In [ ]:
# Compute standard deviations of simulated TFP, output, consumption, investment, and capital
...

In [ ]:
# Compute correlation coefficients of simulated TFP, output, consumption, investment, and capital
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Questions** 

1. In the data simulated from the stochastic Solow growth model, which variables appear to be perfectly correlated?
2. Refer to the Notebook from the class on "Introduction to Business Cycle Data." Compared with the actual correlations, does the stochastic Solow growth model under- or overpredict the correlation between output and consumption over the cycle? What about the correlation between output and investment?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

